#TASK 9 — NLP Project

##Importing Libraries

In [18]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib

##Loading Dataset

In [2]:
# Load the dataset
df = pd.read_csv('/content/patient_feedback_cleaned.csv')

# Display the first few rows of the DataFrame
print(df.head())

           Theme                                        Feedback  Sentiment  \
0      discharge          discharge instructions were very clear          1   
1  communication  i felt the doctor used too much medical jargon          1   
2     medication  i appreciated the reminders about my medicines          1   
3      discharge   instructions after discharge were not helpful          0   
4      discharge   instructions after discharge were not helpful          0   

   Satisfaction  Readmission  
0             4            1  
1             5            0  
2             4            0  
3             1            1  
4             2            1  


## Select relevant columns

For this NLP task, we'll focus on the `Feedback` column for text analysis and the `Sentiment` column as the target variable.

In [3]:
print(df.columns)

Index(['Theme', 'Feedback', 'Sentiment', 'Satisfaction', 'Readmission'], dtype='object')


In [4]:
df_selected = df[['Feedback', 'Sentiment']]
print(df_selected.head())

                                         Feedback  Sentiment
0          discharge instructions were very clear          1
1  i felt the doctor used too much medical jargon          1
2  i appreciated the reminders about my medicines          1
3   instructions after discharge were not helpful          0
4   instructions after discharge were not helpful          0


## Text Preprocessing

To prepare the `Feedback` text for analysis, we'll convert it to lowercase and remove punctuation. This step helps in standardizing the text data.

In [6]:
def preprocess_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    return text

df_selected['Processed_Feedback'] = df_selected['Feedback'].apply(preprocess_text)
print(df_selected.head())

                                         Feedback  Sentiment  \
0          discharge instructions were very clear          1   
1  i felt the doctor used too much medical jargon          1   
2  i appreciated the reminders about my medicines          1   
3   instructions after discharge were not helpful          0   
4   instructions after discharge were not helpful          0   

                               Processed_Feedback  
0          discharge instructions were very clear  
1  i felt the doctor used too much medical jargon  
2  i appreciated the reminders about my medicines  
3   instructions after discharge were not helpful  
4   instructions after discharge were not helpful  


/tmp/ipykernel_13706/1229885984.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Processed_Feedback'] = df_selected['Feedback'].apply(preprocess_text)


## Tokenization and Stopword Removal

After basic cleaning, we'll tokenize the text (break it into individual words) and remove common English stopwords. Stopwords are words like 'the', 'is', 'and', which don't carry much meaning and can be removed to improve model performance.

In [8]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))

def tokenize_and_remove_stopwords(text):
    word_tokens = word_tokenize(text)
    filtered_sentence = [w for w in word_tokens if not w in stop_words]
    return ' '.join(filtered_sentence)

df_selected['Tokenized_Feedback'] = df_selected['Processed_Feedback'].apply(tokenize_and_remove_stopwords)
print(df_selected.head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


                                         Feedback  Sentiment  \
0          discharge instructions were very clear          1   
1  i felt the doctor used too much medical jargon          1   
2  i appreciated the reminders about my medicines          1   
3   instructions after discharge were not helpful          0   
4   instructions after discharge were not helpful          0   

                               Processed_Feedback  \
0          discharge instructions were very clear   
1  i felt the doctor used too much medical jargon   
2  i appreciated the reminders about my medicines   
3   instructions after discharge were not helpful   
4   instructions after discharge were not helpful   

                     Tokenized_Feedback  
0          discharge instructions clear  
1  felt doctor used much medical jargon  
2       appreciated reminders medicines  
3        instructions discharge helpful  
4        instructions discharge helpful  


/tmp/ipykernel_13706/3453285615.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Tokenized_Feedback'] = df_selected['Processed_Feedback'].apply(tokenize_and_remove_stopwords)


## Feature Extraction (TF-IDF)

Now that the text is preprocessed, we need to convert it into numerical features. TF-IDF (Term Frequency-Inverse Document Frequency) is a widely used technique for this, reflecting the importance of a word in a document relative to a corpus.

In [10]:
# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting to 5000 features for practical reasons

# Fit and transform the tokenized feedback
X = tfidf_vectorizer.fit_transform(df_selected['Tokenized_Feedback'])

# Display the shape of the resulting feature matrix
print('Shape of TF-IDF feature matrix:', X.shape)

Shape of TF-IDF feature matrix: (95, 57)


## Data Splitting

Before training a model, we need to split our dataset into training and testing sets. The training set will be used to teach the model, and the testing set will be used to evaluate how well it performs on new, unseen data. We'll use the `Sentiment` column as our target variable (y).

In [12]:
# Define target variable (y)
y = df_selected['Sentiment']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the shapes of the resulting datasets
print('Shape of X_train:', X_train.shape)
print('Shape of X_test:', X_test.shape)
print('Shape of y_train:', y_train.shape)
print('Shape of y_test:', y_test.shape)

Shape of X_train: (76, 57)
Shape of X_test: (19, 57)
Shape of y_train: (76,)
Shape of y_test: (19,)


## Model Training (Logistic Regression)

With our data preprocessed and split, we can now train a machine learning model to predict sentiment. Logistic Regression is a suitable choice for binary classification tasks like sentiment analysis.

In [14]:
# Initialize and train the Logistic Regression model
model = LogisticRegression(random_state=42, solver='liblinear') # Using 'liblinear' solver for small datasets
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


## Model Evaluation

After training, we need to evaluate our model's performance on the test set (`X_test`) using various classification metrics to understand how well it generalizes to new data.

In [17]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-Score: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19



## Save Model and Other Artifacts

It's good practice to save trained models and other important artifacts (like the TF-IDF vectorizer) for future use, so you don't have to retrain or re-process data every time.

In [20]:
#Save the trained Logistic Regression model
joblib.dump(model, 'logistic_regression_sentiment_model.pkl')
print("Logistic Regression model saved as 'logistic_regression_sentiment_model.pkl'")

# Optionally, save the TF-IDF vectorizer as well
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
print("TF-IDF vectorizer saved as 'tfidf_vectorizer.pkl'")

Logistic Regression model saved as 'logistic_regression_sentiment_model.pkl'
TF-IDF vectorizer saved as 'tfidf_vectorizer.pkl'
